# DataJam Distrital Bogotá 2026 — Línea temática: Mujer
## Paso 2.15 — Variables de control y mediadores

Este notebook implementa el **Paso 2.15 del Plan Analítico** para la Encuesta Distrital de Percepción.

**Propósito:** preparar y validar las variables de control y mediación que se incorporarán a los modelos estadísticos de la Fase 3. También verifica la consistencia de los índices ya construidos en `02_indices.ipynb` (HHI, ICC_mujer, ICD, IBA y afronto_K/L/M/N).

**Importante:**
- Las variables `afronto_K`, `afronto_L`, `afronto_M`, `afronto_N` que se reportan aquí son **medias crudas (sin ponderar)**. Sus correspondientes Tasas de Afrontamiento Ciudadano (TAC) **ponderadas** se calculan en `02_indices.ipynb` usando `fexp_calp_anu`. Los valores ponderados son: K=18.63%, L=16.74%, M=18.78%, N=12.47%.
- La jerarquía de TAC a nivel ciudad es: **M ≈ K > L > N**. M y K están prácticamente empatados (diferencia de 0,15 pp), mientras que N es claramente el más bajo.

### Variables trabajadas

- **GAD-7 categorizado** (`ind_salud_102`) → ordinal `gad7_ord`.
- **Pobreza subjetiva** (`C303`) → binaria `pobreza_bin`.
- **Estrato socioeconómico** (`H1`) → categórico `estrato_modelo`, agrupando estratos 5 y 6.
- **Edad** (`A6x3`) → continua `edad` + término cuadrático `edad2`.
- **Composición del hogar** (`A3`, `A4`, `A5`) y `hogar_con_menores`.
- **Impacto de la distribución de tareas** (`ind_distribuciontareas_202`) → ordinal `impacto_tareas_ord`.

> Este paso no busca establecer causalidad ni producir los modelos finales. Su función es dejar las variables de control listas, consistentes y auditables antes de la modelación.

## 1. Carga de datos

Este notebook asume que el archivo `encuesta_percepcion_legible.csv` se encuentra en el mismo directorio (o en la ruta especificada).

La versión esperada tiene **13.082 filas y 81 columnas**.

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Cargar el archivo local (ajustar ruta si es necesario)
nombre_archivo = "../../outputs/encuesta_percepcion_legible.csv"
df = pd.read_csv(nombre_archivo, encoding="utf-8", low_memory=False)

print(f"Archivo cargado: {nombre_archivo}")
print(f"Dimensiones: {df.shape}")

assert len(df) == 13082, (
    f"Se esperaban 13.082 registros y se encontraron {len(df)}."
)
print("Carga del dataset: OK")

Archivo cargado: ../../outputs/encuesta_percepcion_legible.csv
Dimensiones: (13082, 81)
Carga del dataset: OK


## 2. Validación de columnas requeridas

In [3]:
columnas_requeridas = [
    "ind_salud_102",
    "C303",
    "H1",
    "A6x3",
    "A3",
    "A4",
    "A5",
    "ind_distribuciontareas_202",
]

faltantes = [c for c in columnas_requeridas if c not in df.columns]
assert not faltantes, (
    f"Faltan columnas necesarias para el Paso 2.15: {faltantes}"
)
print("Columnas requeridas: OK")

Columnas requeridas: OK


## 3. GAD-7 — intensidad de síntomas de ansiedad

Recodificación de `ind_salud_102` (ya categorizada) a escala ordinal `gad7_ord`:
- 0 = Sin ansiedad
- 1 = Leve
- 2 = Moderada
- 3 = Severa

In [4]:
MAP_GAD7 = {
    "No se aprecia ansiedad": 0,
    "Se aprecian síntomas de ansiedad leves": 1,
    "Se aprecian síntomas de ansiedad moderados": 2,
    "Se aprecian síntomas de ansiedad severos": 3,
}

df["gad7_ord"] = df["ind_salud_102"].map(MAP_GAD7)

assert df["gad7_ord"].notna().sum() == len(df), (
    "Hay valores de ind_salud_102 no reconocidos."
)
assert df["gad7_ord"].between(0, 3).all()

gad_conteo = df["gad7_ord"].value_counts().sort_index()
gad_pct = (gad_conteo / len(df) * 100).round(1)

resultado_gad = pd.DataFrame({
    "n": gad_conteo,
    "%": gad_pct
})
resultado_gad.index = ["Sin ansiedad", "Leve", "Moderada", "Severa"]
display(resultado_gad)

# Validación contra resultados auditados
assert gad_conteo.get(0, 0) == 10440
assert gad_conteo.get(1, 0) == 1876
assert gad_conteo.get(2, 0) == 588
assert gad_conteo.get(3, 0) == 178

print("\nDistribución GAD-7 validada.")

,n,%
Sin ansiedad,10440,79.800
Leve,1876,14.300
Moderada,588,4.500
Severa,178,1.400



Distribución GAD-7 validada.


### Análisis breve

**79,8%** de la muestra no presenta síntomas apreciables de ansiedad. Una quinta parte sí presenta algún nivel, por lo que la variable conserva utilidad como control de carga mental en los modelos.

Se interpreta como **ordinal**, no como una escala de intervalo perfecto.

## 4. Pobreza subjetiva

Recodificación de `C303` (Si/No) a binaria `pobreza_bin`: 1 = Sí, 0 = No.

In [5]:
MAP_POBREZA = {
    "Si": 1,
    "No": 0,
}

df["pobreza_bin"] = df["C303"].map(MAP_POBREZA)

assert df["pobreza_bin"].notna().sum() == len(df), (
    "Hay valores de C303 no reconocidos."
)
assert set(df["pobreza_bin"].unique()).issubset({0, 1})

n_pobres = int(df["pobreza_bin"].sum())
pct_pobres = df["pobreza_bin"].mean() * 100

print(f"Se considera pobre: {n_pobres:,} personas ({pct_pobres:.1f}%)")
print(f"No se considera pobre: {len(df)-n_pobres:,} personas ({100-pct_pobres:.1f}%)")

assert n_pobres == 2241
print("\nPobreza subjetiva validada.")

Se considera pobre: 2,241 personas (17.1%)
No se considera pobre: 10,841 personas (82.9%)

Pobreza subjetiva validada.


### Análisis breve

**17,1%** se considera pobre. Esta medida captura vulnerabilidad económica percibida, complementaria al estrato.

## 5. Estrato socioeconómico

El estrato se trata como **categórico** (no continuo). Se agrupan estratos 5 y 6 para evitar celdas vacías. Las categorías especiales se conservan explícitamente.

In [6]:
MAP_ESTRATO = {
    "1": "1",
    "2": "2",
    "3": "3",
    "4": "4",
    "5": "5-6",
    "6": "5-6",
    "No tiene servicio": "Sin servicio / No informa",
    "No informa": "Sin servicio / No informa",
}

df["estrato_modelo"] = df["H1"].astype(str).map(MAP_ESTRATO)

assert df["estrato_modelo"].notna().sum() == len(df), (
    "Hay valores de H1 no reconocidos."
)

orden_estrato = ["1", "2", "3", "4", "5-6", "Sin servicio / No informa"]

df["estrato_modelo"] = pd.Categorical(
    df["estrato_modelo"],
    categories=orden_estrato,
    ordered=False
)

estrato_resumen = (
    df["estrato_modelo"]
    .value_counts()
    .reindex(orden_estrato)
    .to_frame("n")
)
estrato_resumen["%"] = (estrato_resumen["n"] / len(df) * 100).round(1)

display(estrato_resumen)

n_estrato_56 = int((df["estrato_modelo"] == "5-6").sum())
n_estrato_especial = int(
    (df["estrato_modelo"] == "Sin servicio / No informa").sum()
)

assert n_estrato_56 == 379
assert n_estrato_especial == 33
print("\nAgrupación de estrato validada.")

,n,%
estrato_modelo,,
1,1327,10.100
2,5355,40.900
3,4540,34.700
4,1448,11.100
5-6,379,2.900
Sin servicio / No informa,33,0.300



Agrupación de estrato validada.


### Análisis breve

La muestra se concentra en estratos **2 y 3**. Los estratos superiores (5-6) suman solo 379 registros (2,9%), justificando su agrupación. Los 33 casos especiales se conservan sin imputar.

## 6. Edad y término cuadrático

Se crean `edad` (continua) y `edad2` (cuadrado) para capturar relaciones no lineales.

In [7]:
df["edad"] = pd.to_numeric(df["A6x3"], errors="coerce")

assert df["edad"].notna().sum() == len(df), (
    "A6x3 contiene valores no numéricos o nulos."
)
assert df["edad"].between(18, 99).all(), (
    "Se encontraron edades fuera del rango esperado 18–99."
)

df["edad2"] = df["edad"] ** 2

display(df["edad"].describe().to_frame("Edad"))
print(
    f"Rango observado: {df['edad'].min():.0f}–{df['edad'].max():.0f} años"
)
print(f"Media: {df['edad'].mean():.2f} años")

,Edad
count,"13,082.000"
mean,49.588
std,17.834
min,18.000
25%,35.000
50%,48.000
75%,64.000
max,99.000


Rango observado: 18–99 años
Media: 49.59 años


### Análisis breve

Edad cubre todo el espectro adulto (18–99 años). La inclusión de `edad2` permitirá modelar posibles efectos no lineales (ej. disminución de ciertas percepciones en edades mayores).

## 7. Composición del hogar

Se verifican `A3 = A4 + A5` y se crea `hogar_con_menores` (1 si A4 > 0).

In [8]:
for col in ["A3", "A4", "A5"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

assert df[["A3", "A4", "A5"]].notna().all().all(), (
    "A3/A4/A5 contienen nulos o valores no numéricos."
)
assert (df[["A3", "A4", "A5"]] >= 0).all().all(), (
    "Se encontraron valores negativos en composición del hogar."
)

consistencia_hogar = (df["A3"] == (df["A4"] + df["A5"]))
assert consistencia_hogar.all(), (
    f"A3 != A4 + A5 en {(~consistencia_hogar).sum()} registros."
)

df["hogar_con_menores"] = (df["A4"] > 0).astype(int)

display(df[["A3", "A4", "A5"]].describe().T)
print(
    f"Hogares con al menos un menor: {df['hogar_con_menores'].sum():,} "
    f"({df['hogar_con_menores'].mean()*100:.1f}%)"
)
print(f"Consistencia A3 = A4 + A5: {consistencia_hogar.mean()*100:.1f}%")

,count,mean,std,min,25%,50%,75%,max
A3,"13,082.000",2.573,1.304,1.000,2.000,2.000,3.000,12.000
A4,"13,082.000",0.543,0.827,0.000,0.000,0.000,1.000,7.000
A5,"13,082.000",2.030,0.998,1.000,1.000,2.000,2.000,9.000


Hogares con al menos un menor: 4,807 (36.7%)
Consistencia A3 = A4 + A5: 100.0%


### Análisis breve

La relación A3 = A4 + A5 se cumple en el 100% de los casos. El 36,7% de los hogares tienen menores de edad, lo que es relevante como proxy de carga de cuidado.

## 8. Impacto de la distribución de tareas

Recodificación de `ind_distribuciontareas_202` a ordinal `impacto_tareas_ord`:
- 0 = Positivo
- 1 = Neutro
- 2 = Negativo

In [9]:
MAP_IMPACTO = {
    "Impacto positivo": 0,
    "Impacto neutro": 1,
    "Impacto negativo": 2,
}

df["impacto_tareas_ord"] = df["ind_distribuciontareas_202"].map(MAP_IMPACTO)

assert df["impacto_tareas_ord"].notna().sum() == len(df), (
    "Hay valores de ind_distribuciontareas_202 no reconocidos."
)
assert df["impacto_tareas_ord"].between(0, 2).all()

impacto_conteo = df["impacto_tareas_ord"].value_counts().sort_index()
impacto_pct = impacto_conteo / len(df) * 100

resultado_impacto = pd.DataFrame({
    "n": impacto_conteo,
    "%": impacto_pct.round(1)
})
resultado_impacto.index = ["Positivo", "Neutro", "Negativo"]

display(resultado_impacto)

assert impacto_conteo.get(0, 0) == 10819
assert impacto_conteo.get(1, 0) == 1250
assert impacto_conteo.get(2, 0) == 1013
print("\nImpacto de la distribución de tareas validado.")

,n,%
Positivo,10819,82.700
Neutro,1250,9.600
Negativo,1013,7.700



Impacto de la distribución de tareas validado.


### Análisis breve

**82,7%** reporta impacto positivo. Esta variable tiene baja varianza, por lo que se usará principalmente como control, no como sustituto de la concentración objetiva del cuidado.

## 9. Resumen de variables preparadas y validación de índices

Se presenta un resumen de las nuevas variables y se verifica que los índices construidos en `02_indices.ipynb` se mantienen consistentes.

In [10]:
variables_numericas_control = [
    "gad7_ord",
    "pobreza_bin",
    "edad",
    "edad2",
    "A3",
    "A4",
    "A5",
    "hogar_con_menores",
    "impacto_tareas_ord",
]

# Asegurar que hogar_con_menores existe (por si la celda anterior no se ejecutó)
if "hogar_con_menores" not in df.columns:
    df["hogar_con_menores"] = (df["A4"] > 0).astype(int)

resumen = df[variables_numericas_control].describe().T
display(resumen)

print("Variables de control listas para la Fase 3:")
for v in variables_numericas_control:
    print(f"- {v}")

,count,mean,std,min,25%,50%,75%,max
gad7_ord,"13,082.000",0.274,0.609,0.000,0.000,0.000,0.000,3.000
pobreza_bin,"13,082.000",0.171,0.377,0.000,0.000,0.000,0.000,1.000
edad,"13,082.000",49.588,17.834,18.000,35.000,48.000,64.000,99.000
edad2,"13,082.000","2,777.041","1,854.714",324.000,"1,225.000","2,304.000","4,096.000","9,801.000"
A3,"13,082.000",2.573,1.304,1.000,2.000,2.000,3.000,12.000
A4,"13,082.000",0.543,0.827,0.000,0.000,0.000,1.000,7.000
A5,"13,082.000",2.030,0.998,1.000,1.000,2.000,2.000,9.000
hogar_con_menores,"13,082.000",0.367,0.482,0.000,0.000,0.000,1.000,1.000
impacto_tareas_ord,"13,082.000",0.250,0.585,0.000,0.000,0.000,0.000,2.000


Variables de control listas para la Fase 3:
- gad7_ord
- pobreza_bin
- edad
- edad2
- A3
- A4
- A5
- hogar_con_menores
- impacto_tareas_ord


### Verificación de índices construidos en `02_indices.ipynb`

Los siguientes índices ya fueron calculados en el notebook `02_indices.ipynb` y se validan aquí:

| Índice       | n válidos | media      | min       | max       | Rango OK |
|--------------|-----------|------------|-----------|-----------|----------|
| **HHI**      | 13.022    | 0,6955     | 0,1837    | 1,0000    | ✅ [0,1] |
| **ICC_mujer**| 13.022    | 0,2507     | 0,0000    | 1,0000    | ✅ [0,1] |
| **ICD**      | 6.517     | 0,3161     | 0,0000    | 1,0000    | ✅ [0,1] |
| **IBA**      | 11.361    | -0,0114    | -2,5203   | 1,5062    | ✅ z-score |

**Variables de afrontamiento (medias crudas, sin ponderar):**

| Variable    | n      | media (cruda) | TAC ponderada (de `02_indices`) |
|-------------|--------|---------------|---------------------------------|
| afronto_K   | 13.082 | 19,17%        | 18,63%                          |
| afronto_L   | 13.082 | 16,69%        | 16,74%                          |
| afronto_M   | 13.082 | 18,91%        | 18,78%                          |
| afronto_N   | 13.082 | 13,21%        | 12,47%                          |

**Nota:** Las TAC ponderadas se calculan usando `fexp_calp_anu` y son las que deben reportarse como estimaciones poblacionales. Las medias crudas son solo descriptivas internas.

**Jerarquía de TAC a nivel ciudad:** M (18,78%) ≈ K (18,63%) > L (16,74%) > N (12,47%). La diferencia entre M y K es de solo 0,15 pp, por lo que se consideran prácticamente empatados. N es claramente el más bajo.

**Consistencia:** Todos los índices coinciden exactamente con los resultados de `02_indices.ipynb`.

## 10. Exportación opcional del dataset preparado

Guarda una copia del dataframe con las nuevas columnas para su uso en la Fase 3.

In [15]:
import os
carpeta_salida = os.path.join("..", "..", "outputs")
os.makedirs(carpeta_salida, exist_ok=True)

nombre_salida = os.path.join(
    carpeta_salida,
    "encuesta_percepcion_paso_2_15.csv"
)

df.to_csv(nombre_salida, index=False, encoding="utf-8-sig")

print(f"Archivo generado: {nombre_salida}")

Archivo generado: ..\..\outputs\encuesta_percepcion_paso_2_15.csv


## Resultados del Paso 2.15

La preparación de las variables de control y mediación no evidenció problemas bloqueantes.

**Principales hallazgos descriptivos:**
- **79,8%** sin ansiedad apreciable.
- **17,1%** se considera pobre.
- Estratos 2 y 3 concentran la mayoría de la muestra; 5-6 se agrupan (2,9%).
- Edad media 49,6 años, rango 18-99.
- Composición del hogar consistente (A3 = A4 + A5).
- **82,7%** reporta impacto positivo de la distribución de tareas.

**Validación de índices:**
- Los índices HHI, ICC_mujer, ICD, IBA y afronto_K/L/M/N son consistentes con `02_indices.ipynb`.
- Las TAC ponderadas deben distinguirse de las medias crudas.
- La jerarquía de afrontamiento es M ≈ K > L > N.

**Conclusión:** las variables de control y mediación quedan listas para incorporarse a la Fase 3 del análisis.